# PedRL_debug — Pedagogical RL on DebugBench (Colab)

Port of the PedRL recipe from GSM8K math to **bug fixing** (LeetCode problems
with implanted bugs, python3 subset). The domain hypothesis: debugging is
*reversal-friendly* — the dataset's `bug_explanation` (a witness) converts a
model that can't fix a bug blind into one that can, so the privileged/blind
gap should be larger than in math, and pedagogical distillation of that gap
should beat vanilla GRPO on sample efficiency more clearly.

Run order (everything runs **from the repo root**):
1. smoke test (~15 min, T4) — full-pipeline plumbing check
2. `build-verified` + `probe` — the **hypothesis check comes before any training**
3. `poc` preset (Qwen2.5-Coder-1.5B) — dense-reward pipeline
4. optional: official LeetCode-judge re-grade, ablations

The sparse-reward `hard` preset (Llama-3.2-3B, gated on HF) is deferred
until model access clears — this notebook covers the ungated Qwen presets.

In [ ]:
!nvidia-smi

## 1. Install dependencies

In [ ]:
%pip install -q -U "trl>=0.17.0" "peft>=0.14.0" "datasets>=2.19.0" "accelerate>=0.34.0" "transformers>=4.49.0"
# Colab preinstalls an old torchao that recent peft chokes on; we don't use it
%pip uninstall -q -y torchao

In [ ]:
# HF token: optional — every model used here (Qwen2.5-Coder) is public.
# Set one anyway if you have it (higher rate limits); key icon in the left
# sidebar -> add secret HF_TOKEN.
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN configured — fine, all models here are ungated")

## 2. Get the code

In [ ]:
import os
if os.path.basename(os.getcwd()) == "PedRL":
    !git pull
else:
    if os.path.exists("PedRL"):
        !git -C PedRL pull
    else:
        !git clone https://github.com/dannyyoon0303/PedRL.git
    %cd PedRL
!ls PedRL_debug

## 3. Smoke test (~15 min on T4)

Full pipeline on Qwen2.5-Coder-0.5B: `build-verified` → `eval-base` →
`teacher` → `corpus` → `assimilate` → `eval-student`. The first stage screens
the whole python3 subset against the reconstructed example tests (reference
must pass, buggy code must fail) and writes
`PedRL_debug/verified_debugbench.json` — built once, shared by every preset.

In [ ]:
!python PedRL_debug/run.py all --preset smoke

## 4. The hypothesis check — run BEFORE any real training

The `probe` samples the **same untrained base model** k times blind and k
times with the witness (`bug_explanation`). The headline number of this
variant is the gap. If the privileged pass rate isn't well above blind,
there is no signal to distill and the expensive runs below aren't worth it.

(The smoke test wrote to `outputs_debug/` too, so clean it first;
`verified_debugbench.json` lives outside and is reused.)

In [ ]:
!rm -rf outputs_debug
!python PedRL_debug/run.py build-verified --preset poc
!python PedRL_debug/run.py probe --preset poc

## 5. Analysis helpers

In [ ]:
# shared plot style + helpers (light surface; series colors: fixed categorical order)
import json, os
import matplotlib.pyplot as plt

C1, C2, C3 = "#2a78d6", "#1baf7a", "#d6742a"   # blue, aqua, orange
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e0"

def style_ax(ax, xlabel="", ylabel="", title=""):
    ax.spines[["top", "right"]].set_visible(False)
    ax.spines[["left", "bottom"]].set_color(GRID)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK2, labelsize=9)
    ax.set_xlabel(xlabel, color=INK2, fontsize=10)
    ax.set_ylabel(ylabel, color=INK2, fontsize=10)
    ax.set_title(title, color=INK, fontsize=11, loc="left", pad=10)

def smooth(xs, w=9):
    if len(xs) < w: return xs
    out = []
    for i in range(len(xs)):
        lo = max(0, i - w // 2); hi = min(len(xs), i + w // 2 + 1)
        out.append(sum(xs[lo:hi]) / (hi - lo))
    return out

def _series(ax, x, y, color, label, dy):
    ax.plot(x, y, color=color, linewidth=1, alpha=0.25)
    ys = smooth(y)
    ax.plot(x, ys, color=color, linewidth=2, label=label)
    ax.annotate(label, (x[-1], ys[-1]), xytext=(6, dy),
                textcoords="offset points", color=INK2, fontsize=9, va="center")

def plot_teacher_log(log_path):
    """Surprisal, learnability, and verifier health over teacher training."""
    log = [json.loads(l) for l in open(log_path)]
    x = [r["rollouts"] for r in log]
    has_ped = "mean_gap" in log[0]
    n_panels = 3 if has_ped else 2
    fig, axes = plt.subplots(1, n_panels, figsize=(5.5 * n_panels, 3.8))
    fig.patch.set_facecolor("white")

    if has_ped:
        ax = axes[0]
        _series(ax, x, [r["mean_gap"] for r in log], C1, "mean gap", 6)
        _series(ax, x, [r["mean_max_gap"] for r in log], C2, "max gap", -6)
        style_ax(ax, "teacher rollouts", "surprisal gap d_t (nats)",
                 "Student surprisal of teacher completions")
        ax.legend(frameon=False, fontsize=9, labelcolor=INK2)
        ax.margins(x=0.14)

    ax = axes[1] if has_ped else axes[0]
    if has_ped:
        _series(ax, x, [r["mean_g"] for r in log], C1, "G_spike", 6)
    _series(ax, x, [r["acc"] for r in log], C2, "acc (all tests)", -6)
    style_ax(ax, "teacher rollouts", "score (0-1)", "Learnability and correctness")
    ax.set_ylim(0, 1.02)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2)
    ax.margins(x=0.14)

    # verifier-health panel: echoing the buggy code back earns 0 by
    # construction, but a rising echo_rate means the policy is probing the
    # judge; extract_rate < 1 means completions without a usable code block
    ax = axes[-1]
    _series(ax, x, [r["echo_rate"] for r in log], C3, "echo rate", 6)
    _series(ax, x, [r["extract_rate"] for r in log], C1, "extract rate", -6)
    style_ax(ax, "teacher rollouts", "fraction", "Verifier health")
    ax.set_ylim(-0.02, 1.05)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2)
    ax.margins(x=0.14)
    plt.tight_layout(); plt.show()

def plot_curves(out_dir):
    """Sample efficiency: local-judge pass@1 vs rollouts, PedRL vs vanilla GRPO."""
    fig, ax = plt.subplots(figsize=(7, 4.2))
    fig.patch.set_facecolor("white")

    base_file = os.path.join(out_dir, "eval_base_hard.json")
    if not os.path.exists(base_file):
        base_file = os.path.join(out_dir, "eval_base.json")
    with open(base_file) as f:
        base_acc = json.load(f)["accuracy"]
    ax.axhline(base_acc, color=INK2, linewidth=1.2, linestyle=(0, (4, 4)))
    ax.annotate("base model", (0, base_acc), xytext=(4, 5),
                textcoords="offset points", color=INK2, fontsize=9)

    for i, (name, color, label) in enumerate([
            ("pedrl", C1, "PedRL (teacher rollouts)"),
            ("baseline", C2, "vanilla GRPO")]):
        with open(os.path.join(out_dir, f"curve_{name}.json")) as f:
            pts = json.load(f)["points"]
        xs = [p["rollouts"] for p in pts]
        ys = [p["accuracy"] for p in pts]
        ax.plot(xs, ys, color=color, linewidth=2, marker="o", markersize=6, label=label)
        ax.annotate(label, (xs[-1], ys[-1]), xytext=(8, 6 if i == 0 else -6),
                    textcoords="offset points", color=INK2, fontsize=9, va="center")

    style_ax(ax, "training rollouts", "DebugBench pass@1 (local judge)",
             "Sample efficiency: PedRL vs vanilla GRPO")
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2, loc="lower right")
    ax.margins(x=0.2)
    ax.set_ylim(bottom=0)
    plt.tight_layout(); plt.show()

def print_evals(out_dir):
    """pass@1 per eval file, with per-category / per-difficulty breakdowns."""
    import glob
    for path in sorted(glob.glob(os.path.join(out_dir, "eval_*.json"))):
        if path.endswith("_leetcode.json"):
            continue
        with open(path) as f:
            r = json.load(f)
        cats = "  ".join(f"{c}={v:.2f}" for c, v in r["accuracy_by_category"].items())
        lvls = "  ".join(f"{l}={v:.2f}" for l, v in r["accuracy_by_level"].items())
        print(f"{r['tag']:>26}: pass@1 = {r['accuracy']:.3f}  (n={r['n']})")
        print(f"{'':>28}{cats}")
        print(f"{'':>28}{lvls}")

def plot_probe(out_dir):
    """Premise check: reward density of blind vs privileged sampling."""
    with open(os.path.join(out_dir, "probe.json")) as f:
        p = json.load(f)
    metrics = [("pass1", "pass@1\n(per-sample)"),
               ("any_correct", ">=1 correct\nin group"),
               ("mixed_groups", "mixed groups\n(GRPO-learnable)")]
    series = [("privileged", C1, f"privileged ({p['privileged']}, untrained)"),
              ("blind", C2, "blind (student)")]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    fig.patch.set_facecolor("white")
    w, xs = 0.34, range(len(metrics))
    for si, (cond, color, label) in enumerate(series):
        vals = [p["conditions"][cond][m] for m, _ in metrics]
        pos = [x + (si - 0.5) * (w + 0.04) for x in xs]
        ax.bar(pos, vals, width=w, color=color, label=label, zorder=2)
        for x, v in zip(pos, vals):
            ax.annotate(f"{v:.2f}", (x, v), xytext=(0, 4), textcoords="offset points",
                        ha="center", color=INK2, fontsize=9)
    ax.set_xticks(list(xs))
    ax.set_xticklabels([lbl for _, lbl in metrics], color=INK2, fontsize=9)
    style_ax(ax, "", "fraction",
             f"Reward density on {p['n']} problems ({p['k']} samples each)")
    ax.set_ylim(0, 1.05)
    ax.legend(frameon=False, fontsize=9, labelcolor=INK2)
    plt.tight_layout(); plt.show()

def plot_witness_gap(out_dir):
    """The domain hypothesis, split out: witness gap per bug category and
    per difficulty. Syntax errors should be nearly free given the witness;
    logic/multiple errors are the interesting middle."""
    with open(os.path.join(out_dir, "probe.json")) as f:
        p = json.load(f)
    blind = p["conditions"]["blind"]
    priv = p["conditions"]["privileged"]
    panels = [("pass1_by_category", "bug category"),
              ("pass1_by_level", "difficulty")]
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    fig.patch.set_facecolor("white")
    for ax, (key, xlabel) in zip(axes, panels):
        keys = sorted(set(blind[key]) | set(priv[key]))
        w, xs = 0.34, range(len(keys))
        for si, (vals, color, label) in enumerate([
                (priv[key], C1, "privileged"), (blind[key], C2, "blind")]):
            pos = [x + (si - 0.5) * (w + 0.04) for x in xs]
            ys = [vals.get(k, 0.0) for k in keys]
            ax.bar(pos, ys, width=w, color=color, label=label, zorder=2)
            for x, v in zip(pos, ys):
                ax.annotate(f"{v:.2f}", (x, v), xytext=(0, 3),
                            textcoords="offset points", ha="center",
                            color=INK2, fontsize=8)
        ax.set_xticks(list(xs))
        ax.set_xticklabels(keys, color=INK2, fontsize=9)
        style_ax(ax, xlabel, "pass@1", f"Witness gap by {xlabel}")
        ax.set_ylim(0, 1.05)
        ax.legend(frameon=False, fontsize=9, labelcolor=INK2)
    plt.tight_layout(); plt.show()

In [ ]:
plot_probe("outputs_debug")
plot_witness_gap("outputs_debug")

**Read this before spending GPU-hours.** The condition pedagogical RL
needs is a *gap*, not just a weak student: blind pass@1 low (headroom) AND
privileged pass@1 high (the teacher can bootstrap). Caveat for this preset:
Qwen2.5-Coder is strong on DebugBench, so blind pass@1 will not be low here —
the `poc` run below is a dense-reward plumbing check, not evidence for the
claim. The regime the claim lives in is the sparse `hard` preset (a
non-coder model on problems it fails 0/k blind) — deferred until Llama
access clears.

## 6. Round 1 — dense-reward PoC (`poc` preset)

Qwen2.5-Coder-1.5B, witness = `bug_explanation`. Reward:
`r_ped = R(tests pass) * G_spike(student surprisal)`. `build-verified` is
skipped automatically (the verified set already exists).

In [ ]:
!python PedRL_debug/run.py all --preset poc

## 7. Round-1 results

In [ ]:
print_evals("outputs_debug")

### 7a. Teacher dynamics

What to look for: `acc` high & stable, `mean_g` rising, `mean_gap` falling —
the teacher keeps fixing bugs but learns to *say the fix* in a way the
student finds plausible. In the third panel, `echo_rate` should stay near
zero (echoing the buggy code is unrewarded by construction) and
`extract_rate` near one. The local judge only has ~2–3 example tests and
they are visible in the prompt, so a policy could in principle hardcode them
— if `acc` climbs while the eval numbers don't, re-grade through the
official judge (Section 8).

In [ ]:
plot_teacher_log("outputs_debug/reward_log_teacher.jsonl")

In [ ]:
# peek at a teacher demonstration the student found most learnable
import json
rows = [json.loads(l) for l in open("outputs_debug/distill_corpus.jsonl")]
rows.sort(key=lambda r: -r["g_spike"])
r = rows[0]
print(f"G_spike={r['g_spike']:.3f}\n")
print(r["completion_text"][:1500])

### 7b. Sample efficiency vs vanilla GRPO

In [ ]:
# vanilla-GRPO baseline at the same rollout budget as the teacher (writes checkpoints too)
!python PedRL_debug/run.py baseline-rl --preset poc

In [ ]:
# learning curves: eval pass@1 vs rollouts (writes outputs_debug/curve_*.json)
!python PedRL_debug/run.py curve-baseline --preset poc
!python PedRL_debug/run.py curve-pedrl --preset poc

In [ ]:
plot_curves("outputs_debug")

## 8. Optional: official LeetCode-judge re-grade

The local judge tests ~2–3 example cases; pass rates are upper bounds. This
stage re-grades an existing `eval_<tag>.json` through LeetCode's hidden
suites (~100 tests) and reports the local judge's false-positive rate — the
audit for headline numbers. Never used in training (15 s cooldown per
submission). Needs a `LEETCODE_SESSION` cookie (browser DevTools →
Application → Cookies); store it as a Colab secret, don't paste it here.

In [ ]:
%pip install -q git+https://github.com/GammaTauAI/leetcode-hard-gym.git
import os
from google.colab import userdata
os.environ["LEETCODE_SESSION"] = userdata.get("LEETCODE_SESSION")

!python PedRL_debug/run.py eval-leetcode --preset poc --set eval_tag=base
!python PedRL_debug/run.py eval-leetcode --preset poc --set eval_tag=student

In [ ]:
# how loose are the example tests? local pass@1 vs OJ pass@1 + FP rate
import json, glob
for path in sorted(glob.glob("outputs_debug*/eval_*_leetcode.json")):
    with open(path) as f:
        r = json.load(f)
    print(f"{path}\n  {json.dumps({k: v for k, v in r.items() if not isinstance(v, list)}, indent=2)}")

## 9. Optional: ablations

In [ ]:
# ablation: plain SFT on the same corpus, no surprisal gate
# (writes eval_student_sft_ablation.json)
!python PedRL_debug/run.py assimilate --preset poc --no-gating
!python PedRL_debug/run.py eval-student --preset poc --no-gating

In [ ]:
# ablation: full corrected code as the privileged info (upper bound on the witness)
# !python PedRL_debug/run.py all --preset poc --set privileged=solution

---
Outputs land in `outputs_debug/`; download `probe.json`, `eval_*.json`,
`curve_*.json`, and `reward_log_*.jsonl` before the runtime recycles.
Candidate code runs with a time limit and memory cap but no OS-level
isolation — which is fine here in a Colab container.